# Semantic Search & Re‑ranking with Embeddings + Groq

**Author:** Ibrahim  
**Environment:** Google Colab / Python 3

## Overview
This notebook implements a state‑of‑the‑art semantic search pipeline:
1. **First stage (fast retrieval):** Convert documents and query into embeddings, then use FAISS to find the top‑k most similar documents.
2. **Second stage (re‑ranking):** Pass the candidate documents to Groq’s Llama to re‑order them by relevance to the query.

This hybrid approach gives the speed of vector search with the accuracy of an LLM judge.

## What You Will Build
- A FAISS index from a sample corpus.
- A function to embed queries and retrieve candidates.
- A Groq‑based re‑ranker that returns the best‑matched document.
- Interactive search interface.

## Why This Matters
Semantic search powers modern search engines, RAG systems, and enterprise document retrieval. Adding an LLM re‑ranker significantly boosts precision.

## Requirements
- **Groq API key** (free from [console.groq.com](https://console.groq.com))

---

**© 2026 Ibrahim – Hybrid semantic search with re‑ranking.**

### Install Dependencies

In [1]:
!pip install -q groq sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 29.3 MB/s eta 0:00:00


### Imports & API Key

In [2]:
import numpy as np
import json
from getpass import getpass
from sentence_transformers import SentenceTransformer
import faiss
from groq import Groq

GROQ_API_KEY = getpass("Enter your Groq API key: ")
client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.3-70b-versatile"

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded.")

Enter your Groq API key: ··········


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


### Build Sample Corpus

In [3]:
documents = [
    "The capital of France is Paris. It is known for the Eiffel Tower.",
    "Python is a versatile programming language used in data science and web development.",
    "Today's weather in New York is sunny with a high of 25°C.",
    "Albert Einstein developed the theory of relativity. He won the Nobel Prize in 1921.",
    "Machine learning is a subset of artificial intelligence. It uses data to train models.",
    "Paris is also famous for its croissants and art museums.",
    "In 2024, the Olympic Games will be held in Paris.",
    "AI agents can use tools to solve complex tasks autonomously."
]
doc_ids = [f"doc_{i}" for i in range(len(documents))]

print(f"Corpus contains {len(documents)} documents.")

Corpus contains 8 documents.


### Create FAISS Index

In [4]:
# Compute embeddings for all documents
doc_embeddings = embedder.encode(documents, convert_to_numpy=True)

# Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)   # L2 distance
index.add(doc_embeddings)

print(f"FAISS index created with {index.ntotal} vectors.")

FAISS index created with 8 vectors.


### Embedding‑Based Retrieval

In [5]:
def retrieve_candidates(query, k=5):
    query_emb = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_emb, k)
    candidates = [documents[i] for i in indices[0]]
    scores = distances[0]  # lower is better (L2)
    return candidates, scores

### LLM Re‑ranking

In [6]:
def rerank_with_llm(query, candidates):
    # Build prompt with numbered candidates
    cand_text = "\n".join([f"[{i}] {doc}" for i, doc in enumerate(candidates)])
    prompt = f"""You are a relevance ranking expert. Given the user query, reorder the candidate documents from most relevant to least relevant. Output only the numbers in order, e.g., "2 0 3 1 4".

User query: {query}

Candidates:
{cand_text}

Relevance order (indices separated by spaces):"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=256
    )
    order_str = response.choices[0].message.content.strip()
    # Parse numbers from response
    import re
    indices = [int(x) for x in re.findall(r'\d+', order_str) if int(x) < len(candidates)]
    reranked = [candidates[i] for i in indices]
    return reranked, order_str

### Full Search Pipeline

In [7]:
def semantic_search(query, top_k=5):
    print(f" Query: {query}")
    candidates, scores = retrieve_candidates(query, k=top_k)
    print("\n Embedding‑based retrieval (top 3 with L2 distance):")
    for i, (doc, dist) in enumerate(zip(candidates[:3], scores[:3])):
        print(f"   {i+1}. (dist={dist:.2f}) {doc[:80]}...")

    reranked, order = rerank_with_llm(query, candidates)
    print(f"\n LLM re‑ranking order: {order}")
    print("\n Re‑ranked results (most relevant first):")
    for i, doc in enumerate(reranked):
        print(f"   {i+1}. {doc[:100]}...")
    return reranked

### Test with Example Queries

In [8]:
test_queries = [
    "Tell me about Paris.",
    "What is machine learning?",
    "Who was Einstein?",
    "Programming language for data science"
]

for q in test_queries:
    semantic_search(q, top_k=4)

 Query: Tell me about Paris.

 Embedding‑based retrieval (top 3 with L2 distance):
   1. (dist=0.79) The capital of France is Paris. It is known for the Eiffel Tower....
   2. (dist=0.83) Paris is also famous for its croissants and art museums....
   3. (dist=1.17) In 2024, the Olympic Games will be held in Paris....

 LLM re‑ranking order: 2 0 1 3

 Re‑ranked results (most relevant first):
   1. In 2024, the Olympic Games will be held in Paris....
   2. The capital of France is Paris. It is known for the Eiffel Tower....
   3. Paris is also famous for its croissants and art museums....
   4. Machine learning is a subset of artificial intelligence. It uses data to train models....
 Query: What is machine learning?

 Embedding‑based retrieval (top 3 with L2 distance):
   1. (dist=0.39) Machine learning is a subset of artificial intelligence. It uses data to train m...
   2. (dist=1.40) AI agents can use tools to solve complex tasks autonomously....
   3. (dist=1.50) Python is a versatil

### Interactive Search

In [9]:
print("Semantic Search + Re‑ranking Engine Ready.")
print("Type 'exit' to quit.\n")

while True:
    q = input(" Your query: ").strip()
    if q.lower() == "exit":
        break
    if not q:
        continue
    semantic_search(q, top_k=4)

Semantic Search + Re‑ranking Engine Ready.
Type 'exit' to quit.

 Your query: Tell me about france
 Query: Tell me about france

 Embedding‑based retrieval (top 3 with L2 distance):
   1. (dist=0.86) The capital of France is Paris. It is known for the Eiffel Tower....
   2. (dist=1.04) Paris is also famous for its croissants and art museums....
   3. (dist=1.36) In 2024, the Olympic Games will be held in Paris....

 LLM re‑ranking order: 0 2 1 3

 Re‑ranked results (most relevant first):
   1. The capital of France is Paris. It is known for the Eiffel Tower....
   2. In 2024, the Olympic Games will be held in Paris....
   3. Paris is also famous for its croissants and art museums....
   4. Machine learning is a subset of artificial intelligence. It uses data to train models....
 Your query: exit


### Final Summary

In [11]:
print("Semantic Search & Re‑ranking - COMPLETED")
print("Author: Ibrahim")
print(" Fast retrieval using FAISS + embeddings.")
print(" Re‑ranking with Groq improves relevance.")
print(" Ready to integrate into RAG or search systems.")

Semantic Search & Re‑ranking - COMPLETED
Author: Ibrahim
 Fast retrieval using FAISS + embeddings.
 Re‑ranking with Groq improves relevance.
 Ready to integrate into RAG or search systems.
